In [86]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,root_mean_squared_error,mean_absolute_error
import os
from tqdm import tqdm


In [87]:
os.chdir('D:\Machine Learning\Datasets')

In [88]:
df=pd.read_csv('Housing.csv')

In [89]:
from sklearn.compose import make_column_selector
ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas') # one hot encoder
col_trnf=ColumnTransformer([('OHE',ohe,make_column_selector(dtype_include=object))],remainder='passthrough',verbose_feature_names_out=True)
col_trnf=col_trnf.set_output(transform='pandas')
X=col_trnf.fit_transform(df)

# Ridge


In [90]:
from sklearn.linear_model import Ridge


In [91]:
boston = pd.read_csv("boston.csv")
boston

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [92]:
X=boston.drop('medv',axis=1)
y=boston['medv']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [93]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
results=[]

In [94]:
# train ridge model with different alpha values
from sklearn.metrics import mean_squared_error


for alpha in alphas:
    model=Ridge(alpha=alpha)
    model.fit(X_train,y_train)

    #Predictions
    y_pred_train=model.predict(X_train)
    y_pred_test=model.predict(X_test)

   # Metrics
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    results.append({
        'alpha': alpha,
        'coef': model.coef_[0],
        'intercept': model.intercept_,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2
    })
    
    print(f"Alpha: {alpha:7.3f} | Coef: {model.coef_[0]:6.3f} | "
          f"Intercept: {model.intercept_:6.3f} | "
          f"Train MSE: {train_mse:6.3f} | Test MSE: {test_mse:6.3f}")


Alpha:   0.001 | Coef: -0.133 | Intercept: 31.621 | Train MSE: 22.545 | Test MSE: 21.518
Alpha:   0.010 | Coef: -0.133 | Intercept: 31.534 | Train MSE: 22.546 | Test MSE: 21.524
Alpha:   0.100 | Coef: -0.133 | Intercept: 30.735 | Train MSE: 22.551 | Test MSE: 21.585
Alpha:   1.000 | Coef: -0.128 | Intercept: 26.765 | Train MSE: 22.715 | Test MSE: 22.044
Alpha:  10.000 | Coef: -0.127 | Intercept: 24.976 | Train MSE: 23.211 | Test MSE: 22.695
Alpha: 100.000 | Coef: -0.130 | Intercept: 36.100 | Train MSE: 24.829 | Test MSE: 23.468
Alpha: 1000.000 | Coef: -0.117 | Intercept: 39.596 | Train MSE: 29.511 | Test MSE: 27.100


Hyperparameter Tuning

In [95]:
alphas = [0.001, 0.01, 0.1, 1, 1.5,2.5,5,10]
scores=[]
for alpha in alphas:
    model=Ridge(alpha=alpha)
    model.fit(X_train,y_train)
        #Predictions
    y_pred=model.predict(X_test)
    scores.append([alpha,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['alpha','score'])
df_scores.sort_values('score')

,alpha,score
2,0.100,3.162397
1,0.010,3.162624
0,0.001,3.162698
3,1.000,3.178539
4,1.500,3.189260
5,2.500,3.204128
6,5.000,3.227276
7,10.000,3.249170


Housing_dataset

In [96]:
housing=pd.read_csv('Housing.csv')

In [97]:
# column transformer
cat_hous=housing[housing.columns[housing.dtypes==object]] # seperating catogorical data
num_hous=housing[housing.columns[housing.dtypes!=object]]
ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas') # one hot encoder
ohe_house=ohe.fit_transform(cat_hous) # cat to 1,0
house=pd.concat([num_hous,ohe_house],axis=1) # row-wise concat
X,y=house.drop('price',axis=1),house['price']


In [107]:
# train-test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
np.linspace(.001,15,30)


array([1.00000000e-03, 5.18206897e-01, 1.03541379e+00, 1.55262069e+00,
       2.06982759e+00, 2.58703448e+00, 3.10424138e+00, 3.62144828e+00,
       4.13865517e+00, 4.65586207e+00, 5.17306897e+00, 5.69027586e+00,
       6.20748276e+00, 6.72468966e+00, 7.24189655e+00, 7.75910345e+00,
       8.27631034e+00, 8.79351724e+00, 9.31072414e+00, 9.82793103e+00,
       1.03451379e+01, 1.08623448e+01, 1.13795517e+01, 1.18967586e+01,
       1.24139655e+01, 1.29311724e+01, 1.34483793e+01, 1.39655862e+01,
       1.44827931e+01, 1.50000000e+01])

In [108]:
alphas = np.linspace(.001,15,30)
scores=[]
for a in alphas:
    model=Ridge(alpha=a)
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    scores.append([a,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['alpha','score'])
df_scores.sort_values('score')

,alpha,score
7,3.621448,11462.396390
8,4.138655,11462.418004
9,4.655862,11462.875243
6,3.104241,11463.351288
10,5.173069,11464.669465
11,5.690276,11466.809072
5,2.587034,11468.848793
12,6.207483,11469.034580
13,6.724690,11471.335969
4,2.069828,11475.110378


In [109]:
# auto generate alphas values
np.arange(0.001,15,.05)

array([1.0000e-03, 5.1000e-02, 1.0100e-01, 1.5100e-01, 2.0100e-01,
       2.5100e-01, 3.0100e-01, 3.5100e-01, 4.0100e-01, 4.5100e-01,
       5.0100e-01, 5.5100e-01, 6.0100e-01, 6.5100e-01, 7.0100e-01,
       7.5100e-01, 8.0100e-01, 8.5100e-01, 9.0100e-01, 9.5100e-01,
       1.0010e+00, 1.0510e+00, 1.1010e+00, 1.1510e+00, 1.2010e+00,
       1.2510e+00, 1.3010e+00, 1.3510e+00, 1.4010e+00, 1.4510e+00,
       1.5010e+00, 1.5510e+00, 1.6010e+00, 1.6510e+00, 1.7010e+00,
       1.7510e+00, 1.8010e+00, 1.8510e+00, 1.9010e+00, 1.9510e+00,
       2.0010e+00, 2.0510e+00, 2.1010e+00, 2.1510e+00, 2.2010e+00,
       2.2510e+00, 2.3010e+00, 2.3510e+00, 2.4010e+00, 2.4510e+00,
       2.5010e+00, 2.5510e+00, 2.6010e+00, 2.6510e+00, 2.7010e+00,
       2.7510e+00, 2.8010e+00, 2.8510e+00, 2.9010e+00, 2.9510e+00,
       3.0010e+00, 3.0510e+00, 3.1010e+00, 3.1510e+00, 3.2010e+00,
       3.2510e+00, 3.3010e+00, 3.3510e+00, 3.4010e+00, 3.4510e+00,
       3.5010e+00, 3.5510e+00, 3.6010e+00, 3.6510e+00, 3.7010e

In [ ]:
np.linspace(.001,1.5,25)

array([1.00000000e-03, 6.34583333e-02, 1.25916667e-01, 1.88375000e-01,
       2.50833333e-01, 3.13291667e-01, 3.75750000e-01, 4.38208333e-01,
       5.00666667e-01, 5.63125000e-01, 6.25583333e-01, 6.88041667e-01,
       7.50500000e-01, 8.12958333e-01, 8.75416667e-01, 9.37875000e-01,
       1.00033333e+00, 1.06279167e+00, 1.12525000e+00, 1.18770833e+00,
       1.25016667e+00, 1.31262500e+00, 1.37508333e+00, 1.43754167e+00,
       1.50000000e+00])

In [110]:
# Salraies

In [119]:
sal=pd.read_csv('Exp_salaries.csv')
sal

,Salary,Years_Previous_Experience,Years Employed,Years_Education,Gender,Department,Number_Supervised
0,32782,1,0,7,Male,A,0
1,32920,3,15,9,Female,A,4
2,29548,6,5,1,Male,A,0
3,39828,6,18,5,Female,A,5
4,31528,11,3,3,Male,A,6
5,38985,7,18,9,Male,A,5
6,41889,16,22,7,Male,A,7
7,38791,4,21,5,Male,A,9
8,28985,1,0,4,Female,A,4
9,24749,2,6,0,Female,A,1


In [123]:
cat_sal=sal[sal.columns[sal.dtypes==object]]
num_sal=sal[sal.columns[sal.dtypes==int]]
ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas')
one_sal=ohe.fit_transform(cat_sal)
sal=pd.concat([cat_sal,num_sal],axis=1)
X,y=sal.drop('Salary',axis=1),sal['Salary']

In [ ]:
for a in alphas:
    model=Ridge(alpha=a)
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    scores.append([a,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['alpha','score'])
df_scores.sort_values('score')

In [124]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


alphas=np.linspace(.001,15,30)
scores=[]
for a in alphas:
    model=Ridge(alpha=a)
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    scores.append([a,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['alpha','score'])
df_scores.sort_values('Score')